# 09 · Nonlinear Geometry Search

*Separate nonlinear geometry from linear slip with variable projection.*

**Learning objectives**

- derive the projected geometry objective
- map parameter trade-offs and local minima
- combine a coarse scan with bounded local refinement
- distinguish optimizer curvature from full geometry uncertainty

**Prerequisites:** Chapters 03–04 and 08  
**Estimated time:** about 60 minutes

Notation follows the [glossary](../docs/glossary.md); axes, units, signs, and
ordering follow [GeoDef conventions](../docs/conventions.md).


## Motivation

Slip is linear only after geometry is fixed. Depth, dip, location, length, and
width alter every Green's-function column nonlinearly. A wrong geometry can be
compensated by wrong slip and still fit the observations, so geometry search
must expose trade-offs rather than return only one optimizer row.


## Theory deepening: variable projection and identifiability

Geometry enters through $G(\theta)$, so define the projected objective

$$\Phi_p(\theta)=\min_m\Phi(\theta,m).$$

Every trial geometry uses the linear solver already developed; only the small
outer parameter vector is nonlinear. This is variable projection. It is more
stable than treating hundreds of slip values and a few geometry values as one
undifferentiated nonlinear vector.

Depth, width, and slip can trade off because a deeper or larger source can
produce a similar broad surface field. A two-dimensional objective surface is
therefore more informative than one optimizer covariance. Coarse scans expose
basins; bounded local refinement locates a minimum within one basin.


## Why geometry is nonlinear

The Green's matrix depends on the geometry parameters $\boldsymbol\theta$
(position, strike, dip, size). Slip is linear *given* the geometry, but the
geometry itself enters through $G(\boldsymbol\theta)$:

$$ \mathbf d = G(\boldsymbol\theta)\,\mathbf m + \boldsymbol\varepsilon. $$

There is no closed form for $\boldsymbol\theta$; we must **search**.

### Separable structure (variable projection)
The key simplification: for *any* trial geometry $\boldsymbol\theta$, recovering
$\mathbf m$ is still the linear, regularized inversion of notebooks 03–08. So we
only search over the handful of nonlinear parameters, and at each trial we solve
the fast linear problem inside:

$$ \Phi(\boldsymbol\theta) = \min_{\mathbf m}
\lVert G(\boldsymbol\theta)\mathbf m - \mathbf d\rVert^2_{C_d}
+ \lambda\lVert L\mathbf m\rVert^2. $$

$\Phi(\boldsymbol\theta)$ is the misfit *after* the best slip has been fit. We
minimize this reduced objective over $\boldsymbol\theta$ alone. Its surface can
have local minima, so a coarse **grid search** before gradient refinement is
good practice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.matrix(
    fault, geodef.data.gnss(lon=glon, lat=glat, east=_zero, north=_zero, up=_zero, sigma_east=_one, sigma_north=_one, sigma_up=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.data.gnss(
    lon=glon, lat=glat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_sta, sigma), sigma_north=np.full(n_sta, sigma), sigma_up=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

The setup fault has a true dip of 15°. Pretend we don't know it. Define the
reduced objective: rebuild the fault at a trial dip, run the linear inversion,
and return its reduced $\chi^2$.

In [ ]:
def make_fault(dip):
    return geodef.Fault.planar(
        lat=-2.0, lon=100.0, depth=25e3, strike=315.0, dip=dip,
        length=180e3, width=90e3, n_length=12, n_width=6,
    )

def misfit(dip):
    r = geodef.solve(make_fault(dip), gnss, components="dip",
                      regularization="laplacian", regularization_strength=1.0)
    return r.reduced_chi2

## Grid search over dip

Scan a range of dips. The misfit is minimized near the true value — and note the
asymmetry: too-steep dips are penalized more sharply than too-shallow ones.

In [ ]:
dips = np.arange(5.0, 41.0, 2.5)
phi = np.array([misfit(d) for d in dips])
best_grid = dips[np.argmin(phi)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dips, phi, "o-")
ax.axvline(15.0, color="green", ls="--", label="true dip 15")
ax.axvline(best_grid, color="red", ls=":", label=f"grid best {best_grid:g}")
ax.set_xlabel("trial dip (deg)"); ax.set_ylabel("reduced chi^2")
ax.set_title("Misfit vs. fault dip"); ax.legend()
print(f"grid-search best dip: {best_grid:g} deg")

## Refine with an optimizer

Starting near the grid minimum, a 1-D bounded optimizer polishes the estimate.
For several geometry parameters you would use `scipy.optimize.minimize` over a
vector $\boldsymbol\theta$; the idea is identical.

In [ ]:
from scipy.optimize import minimize_scalar

opt = minimize_scalar(misfit, bounds=(8.0, 25.0), method="bounded")
print(f"refined dip:  {opt.x:.2f} deg   (true 15.0)")
print(f"reduced chi^2 at optimum: {opt.fun:.3f}")

## Recovered geometry and slip

At the recovered dip, the slip inversion returns a sensible bump — geometry and
slip together explain the data.

In [ ]:
best_fault = make_fault(opt.x)
best = geodef.solve(best_fault, gnss, components="dip",
                     regularization="laplacian", regularization_strength=1.0)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=ax1, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True (dip 15)",
                 colorbar_label="Slip (m)")
geodef.plot.slip(best_fault, best.dip_slip, ax=ax2, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title=f"Recovered (dip {opt.x:.1f})",
                 colorbar_label="Slip (m)")
plt.tight_layout()

## Checkpoint questions

**Why minimize over slip inside each geometry trial?**

<details><summary>Answer</summary>



Slip is linear and can be solved exactly, leaving a much smaller nonlinear search.



</details>

**What does a long misfit valley mean?**

<details><summary>Answer</summary>



Different geometries can be compensated by slip and are not separately identifiable.



</details>

**Why scan before local optimization?**

<details><summary>Answer</summary>



A scan reveals multiple basins and boundary behavior.



</details>


## Common mistakes

- Reporting only optimizer success hides starting-point and basin dependence.
- Comparing geometries with inconsistent regularization changes both geometry and prior.
- Interpreting local Hessian errors across a multimodal surface is invalid.


## Recap

- Geometry makes $G$ nonlinear while slip remains linear conditionally.
- Variable projection exploits that separable structure.
- Objective surfaces and restarts are essential evidence for identifiability.

Return to the [workflow decision guides](../docs/workflow.md) when adapting
this method. The course map in [the tutorial README](README.md) identifies the
next chapter and optional branches.


## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are published separately in `solutions/09_nonlinear_geometry_solutions.ipynb`.

1. Scan dip, then refine from both sides of the minimum.
2. Start from a deliberately poor basin and document the result.
3. Repeat the depth scan after doubling noise.
4. Challenge: map depth versus width and explain the trade-off valley physically.


## Further reading

- Golub & Pereyra (1973), variable projection.
- Tarantola (2005), nonlinear inverse problems.
- Segall (2010), geometry–slip trade-offs.
